<a href="https://colab.research.google.com/github/eschain93/ban5600_800/blob/main/week3/Homework_3_Part_2_Chainani_Emma.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part 2

## Google Authentication

In [ ]:
from google.colab import auth
import gspread
from google.auth import compute_engine
from google.auth import default

# Authenticate the user with Google Colab
auth.authenticate_user()
# Get default credentials for Google services
creds, _ = default()
# Authorize gspread (Google Sheets API) with the obtained credentials
gc = gspread.authorize(creds)

print("Successfully authenticated!")

Successfully authenticated!


## Excercise

In [ ]:
import gspread.exceptions

# --- 1. Create or Open a Sheet ---

sheet_name = 'Publishing Data'

try:
    sh = gc.open(sheet_name)
    print(f"Opening existing Google Sheet: '{sh.title}'")
except gspread.exceptions.SpreadsheetNotFound:
    sh = gc.create(sheet_name)
    print(f"Created new Google Sheet: '{sh.title}'")
    # Set sharing permissions for the newly created sheet
    sh.share(None, perm_type='anyone', role='reader')
    print(f"Sharing permissions set for '{sh.title}': Anyone with the link has viewer access.")

# Get the first worksheet in the spreadsheet
worksheet = sh.get_worksheet(0)

# --- 2. Define the Data ---
# Header Row for the report
headers = ["article_url", "publish_date", "topic_cluster","status"]

# Get current values of the first row
current_first_row = worksheet.row_values(1)

# Check if the current first row matches the desired headers
if current_first_row != headers:
    print("Headers do not match or are missing. Replacing row 1 with new headers.")
    # Clear the entire first row to ensure a clean slate
    worksheet.batch_clear(["1:1"]) # Correctly clears cells A1 to the last column in row 1
    # Update the first row with the new headers
    worksheet.update(range_name='A1', values=[headers])
else:
    print("Headers already exist and match the desired format. Skipping update.")

print("Google Sheet link:", sh.url)

Created new Google Sheet: 'Publishing Data'
Sharing permissions set for 'Publishing Data': Anyone with the link has viewer access.
Headers do not match or are missing. Replacing row 1 with new headers.
Google Sheet link: https://docs.google.com/spreadsheets/d/18MzBNpprL56m9lVwFZgJPV3ZvRAMn523ngk8x_NtHgk


In [ ]:
import gspread

# Sample Data to write to the sheet
article_data = [
    ["https://example.com/ai-trends", "2026-01-15", "Artificial Intelligence", "Published"],
    ["https://example.com/data-science-tips", "2026-04-12", "Data Science", "Published"],
    ["https://example.com/cloud-computing", "", "Cloud", "Review"],
    ["https://example.com/python-basics", "2026-03-05", "Programming", "Published"],
    ["https://example.com/machine-learning", "", "Artificial Intelligence", "Draft"],
    ["https://example.com/types-of-machine-learning", "2026-04-10", "Artificial Intelligence", "Published"],
    ["https://example.com/python-for-experts", "2026-04-12", "Programming", "Published"]
]

# --- 3. Loop through and write ---
print("Writing data to Google Sheets...")

# Get all existing values from the worksheet (including headers to find actual row numbers)
all_sheet_values = worksheet.get_all_values()
# Create a dictionary to quickly look up article URLs and their actual row numbers
# Row numbers in gspread are 1-based, and we assume article_url is the first column (index 0)
existing_urls_map = {row[0]: i + 1 for i, row in enumerate(all_sheet_values) if i > 0 and row[0]}

for row_data in article_data:
    article_url = row_data[0]
    if article_url in existing_urls_map:
        row_number_to_update = existing_urls_map[article_url]
        current_row_data = worksheet.row_values(row_number_to_update)

        # Convert row_data to match the format of current_row_data for comparison
        # gspread returns values as strings, so we ensure comparison is fair.
        str_row_data = [str(item) for item in row_data]

        if str_row_data != current_row_data:
            # Article exists and data has changed, update the row
            worksheet.update(values=[row_data], range_name=f'A{row_number_to_update}') # Fixed DeprecationWarning
            print(f"Updated existing row (at row {row_number_to_update}): {row_data}")
        else:
            print(f"Skipping existing row (no changes at row {row_number_to_update}): {row_data}")
    else:
        # Article does not exist, append as a new row
        worksheet.append_row(row_data)
        print(f"Appended new row: {row_data}")

print(f"Done! You can find your sheet at: {sh.url}")

Writing data to Google Sheets...
Skipping existing row (no changes at row 2): ['https://example.com/ai-trends', '2026-01-15', 'Artificial Intelligence', 'Published']
Updated existing row (at row 3): ['https://example.com/data-science-tips', '2026-04-12', 'Data Science', 'Published']
Skipping existing row (no changes at row 4): ['https://example.com/cloud-computing', '', 'Cloud', 'Review']
Skipping existing row (no changes at row 5): ['https://example.com/python-basics', '2026-03-05', 'Programming', 'Published']
Skipping existing row (no changes at row 6): ['https://example.com/machine-learning', '', 'Artificial Intelligence', 'Draft']
Skipping existing row (no changes at row 7): ['https://example.com/types-of-machine-learning', '2026-04-10', 'Artificial Intelligence', 'Published']
Skipping existing row (no changes at row 8): ['https://example.com/python-for-experts', '2026-04-12', 'Programming', 'Published']
Done! You can find your sheet at: https://docs.google.com/spreadsheets/d/18MzB

## OpenAI API

In [ ]:
import os
from google.colab import userdata
from openai import OpenAI

# 1. Pull the OpenAI API key from Colab Secrets
os.environ["OPENAI_API_KEY"] = userdata.get('openaikey')

# 2. Initialize the OpenAI client
client = OpenAI()

# 3. Use the article data in the prompt to generate a summary
response = client.chat.completions.create(
    model="gpt-4o", # Specifies the AI model to use
    messages=[
        {"role": "user", "content": f"""
        Interpret this publishing report data set.
        Format: [article_url, publish_date, topic_cluster, status]

        data = {article_data}

        Please provide a summary of the publishing status and any trends you see in the topic clusters.
        """}
    ]
)

# Extract the AI-generated summary content
full_summary_content = response.choices[0].message.content
print(full_summary_content)

# Add the response content to a new tab in the Google Sheet
# Check if a worksheet named 'Summary' already exists
existing_worksheets = sh.worksheets()
summary_worksheet = None
for ws in existing_worksheets:
    if ws.title == 'Summary':
        summary_worksheet = ws
        break

if summary_worksheet is None:
    # Create a new worksheet named 'Summary' if it doesn't exist
    summary_worksheet = sh.add_worksheet(title='Summary', rows='100', cols='20')
    print("Created new worksheet: 'Summary'")
else:
    # If the worksheet exists, clear its existing content to update it
    summary_worksheet.clear()
    print("Cleared existing content in 'Summary' worksheet.")

# Split the summary into sections based on '###' headers for structured writing
sections = [s.strip() for s in full_summary_content.split('###') if s.strip()]

current_row = 1 # Start writing from the first row
for section in sections:
    # Each section is expected to start with a header, followed by content.
    # Find the first newline to separate the header from its body
    first_newline = section.find('\n')

    if first_newline != -1:
        header = section[:first_newline].strip() # Extract header
        body = section[first_newline:].strip()   # Extract body
    else:
        header = section.strip() # If no newline, the whole section is a header
        body = ""

    if header:
        # Write the header to a cell in column A
        summary_worksheet.update(range_name=f'A{current_row}', values=[[header]]) # Fixed DeprecationWarning
        current_row += 1 # Move to the next row for the body or next section

    if body:
        # Write the body content to a cell below the header
        # The entire body is placed in a single cell for the section.
        summary_worksheet.update(range_name=f'A{current_row}', values=[[body]]) # Fixed DeprecationWarning
        current_row += 1 # Move to the next row

    # Add an empty row for visual separation between different sections
    current_row += 1

print(f"Response content added to Google Sheet: {sh.url} (tab 'Summary') with sections formatted.")

The provided data set contains seven articles with details about their URLs, publish dates, topic clusters, and publication status. Here's a summary of the publishing status and observed trends in the topic clusters:

### Publishing Status Summary:
1. **Published Articles:**
   - There are five articles that have been marked as "Published."
   - Published articles and their publish dates are:
     - Artificial Intelligence: 
       - https://example.com/ai-trends (2026-01-15)
       - https://example.com/types-of-machine-learning (2026-04-10)
     - Data Science:
       - https://example.com/data-science-tips (2026-04-12)
     - Programming:
       - https://example.com/python-basics (2026-03-05)
       - https://example.com/python-for-experts (2026-04-12)

2. **Articles Under Review:**
   - One article is currently in "Review" status:
     - Cloud: https://example.com/cloud-computing

3. **Draft Articles:**
   - There is one article currently in "Draft" status:
     - Artificial Intel

In [ ]:
# Print the raw OpenAI API response object
print(response)

ChatCompletion(id='chatcmpl-DTvTouzYPwjCdhyZPyhVhtHvrTCrR', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='The provided data set contains seven articles with details about their URLs, publish dates, topic clusters, and publication status. Here\'s a summary of the publishing status and observed trends in the topic clusters:\n\n### Publishing Status Summary:\n1. **Published Articles:**\n   - There are five articles that have been marked as "Published."\n   - Published articles and their publish dates are:\n     - Artificial Intelligence: \n       - https://example.com/ai-trends (2026-01-15)\n       - https://example.com/types-of-machine-learning (2026-04-10)\n     - Data Science:\n       - https://example.com/data-science-tips (2026-04-12)\n     - Programming:\n       - https://example.com/python-basics (2026-03-05)\n       - https://example.com/python-for-experts (2026-04-12)\n\n2. **Articles Under Review:**\n   - One article is curr

In [ ]:
from pprint import pprint

# Pretty-print the full OpenAI API response object for better readability
# Convert the ChatCompletion object to a dictionary for more structured pretty-printing
pprint(response.model_dump())

{'choices': [{'finish_reason': 'stop',
              'index': 0,
              'logprobs': None,
              'message': {'annotations': [],
                          'audio': None,
                          'content': 'The provided data set contains seven '
                                     'articles with details about their URLs, '
                                     'publish dates, topic clusters, and '
                                     "publication status. Here's a summary of "
                                     'the publishing status and observed '
                                     'trends in the topic clusters:\n'
                                     '\n'
                                     '### Publishing Status Summary:\n'
                                     '1. **Published Articles:**\n'
                                     '   - There are five articles that have '
                                     'been marked as "Published."\n'
                             

In [ ]:
#Print Final Google Sheet URL
print("Google Sheet link:", sh.url)

Google Sheet link: https://docs.google.com/spreadsheets/d/18MzBNpprL56m9lVwFZgJPV3ZvRAMn523ngk8x_NtHgk
